# M05b · Fonctions de plusieurs variables et gradient — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m05b_gradient/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m05b_gradient/03_notebook.ipynb`, créée par `python tools/start_chapter.py M05b`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M05b.1 à M05b.10, M05b.14, M05b.16 | Les exercices papier (vérification) | ✏️ 🔗 📈 | | |
| A | M05b.18 | Une dérivée partielle par différence centrée · `partial_derivative` | 🔨 | ★ | 20 |
|  | M05b.19 | Dérivée directionnelle et direction de plus grande pente · `directional_derivative, steepest_ascent_direction` | 🔨 | ★★ | 20 |
|  | M05b.20 | Le plan tangent · `tangent_plane` | 🔨 | ★ | 15 |
|  | M05b.21 | SymPy : dérivées partielles, gradient, points critiques | 📦 | ★ | 20 |
| B | M05b.22 | Prédire : la dérivée directionnelle le long d'une ligne de niveau | 🔮 | ★ | 10 |
|  | M05b.23 | L'erreur du plan tangent quand on s'éloigne du point | 🔬 | ★★ | 20 |
|  | M05b.24 | Lignes de niveau et flèches du gradient sur la même figure | 🎨 | ★ | 20 |
| C | M05b.25 | Chasse à l'erreur : la mauvaise variable dérivée, une direction non unitaire | 🐛 | ★ | 15 |
|  | M05b.26 | Vectoriser : une fonction de deux variables sur une grille (`np.meshgrid`) | 🛠️ | ★ | 15 |
|  | M05b.27 | Défi : retrouver une fonction du second degré à partir de son gradient | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M05b")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : lignes de niveau, dérivées partielles, gradient, dérivée directionnelle, plus grande pente, plan tangent, dérivées secondes, points critiques, pas de descente, argmin sur un segment, gradient d'une loss, lecture d'une carte.
- Écrire les quatre fonctions de `mymath/multivar.py` et les tester, puis utiliser SymPy.
- Prédire la pente le long d'une ligne de niveau, mesurer l'erreur du plan tangent, tracer une carte avec les flèches du gradient, chasser des erreurs, vectoriser une grille, retrouver une fonction à partir de son gradient.

**Rappel express.** $\nabla f = (f_x, f_y)$ ; $D_{\mathbf{u}} f = \nabla f \cdot \mathbf{u}$ avec $\|\mathbf{u}\| = 1$ ; plus grande montée $\frac{\nabla f}{\|\nabla f\|}$, de pente $\|\nabla f\|$ ; plan tangent $z = f(a, b) + f_x\,(x - a) + f_y\,(y - b)$ ; $D = f_{xx}f_{yy} - f_{xy}^2$ : $D > 0$ et $f_{xx} > 0$ minimum, $D > 0$ et $f_{xx} < 0$ maximum, $D < 0$ selle ; descente : $\mathbf{x} - \eta\,\nabla f(\mathbf{x})$.

## Partie 0 · Vérifier tes exercices papier (✏️ M05b.1 à M05b.10, 🔗 M05b.14, 📈 M05b.16)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"7/9"` ; une expression en $x$ et $y$ entre guillemets, `"3*x**2*y - y"`, avec `exp(u)` pour $e^u$ et `sqrt(u)` pour $\sqrt{u}$ ;
- une liste `[-1.5, 2.5]` (un gradient, un point, un vecteur), `True` / `False`, ou une lettre entre guillemets, `"B"`.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(3, 4) is accepted too, as "3/4"

**Ex M05b.1 — Les lignes de niveau de trois fonctions**

In [ ]:
# Ex M05b.1: the values you found on paper
answer_M05b_1a = ...  # a) an integer
answer_M05b_1b = ...  # b) an integer
answer_M05b_1c = ...  # c) an integer
answer_M05b_1d = ...  # d) an integer
answer_M05b_1e = ...  # e) an integer
answer_M05b_1f = ...  # f) an integer
answer_M05b_1g = ...  # g) True or False
answer_M05b_1h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M05b_1a, answer_M05b_1b, answer_M05b_1c, answer_M05b_1d, answer_M05b_1e, answer_M05b_1f, answer_M05b_1g, answer_M05b_1h]):
    wbm.check(f"M05b.1{letter}", answer)

**Ex M05b.2 — Dérivées partielles**

In [ ]:
# Ex M05b.2: the values you found on paper
answer_M05b_2a = ...  # a) an expression in x and y, as text
answer_M05b_2b = ...  # b) an expression in x and y, as text
answer_M05b_2c = ...  # c) an integer
answer_M05b_2d = ...  # d) an integer
answer_M05b_2e = ...  # e) an integer
answer_M05b_2f = ...  # f) an integer
answer_M05b_2g = ...  # g) an integer
answer_M05b_2h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M05b_2a, answer_M05b_2b, answer_M05b_2c, answer_M05b_2d, answer_M05b_2e, answer_M05b_2f, answer_M05b_2g, answer_M05b_2h]):
    wbm.check(f"M05b.2{letter}", answer)

**Ex M05b.3 — Le gradient en un point et sa norme**

In [ ]:
# Ex M05b.3: the values you found on paper
answer_M05b_3a = ...  # a) a list of 2 integers
answer_M05b_3b = ...  # b) a number (4 decimals)
answer_M05b_3c = ...  # c) a list of 2 integers
answer_M05b_3d = ...  # d) a list (3 decimals)
answer_M05b_3e = ...  # e) a list (4 decimals)
answer_M05b_3f = ...  # f) a number (4 decimals)

for letter, answer in zip("abcdef", [answer_M05b_3a, answer_M05b_3b, answer_M05b_3c, answer_M05b_3d, answer_M05b_3e, answer_M05b_3f]):
    wbm.check(f"M05b.3{letter}", answer)

**Ex M05b.4 — La dérivée directionnelle dans plusieurs directions**

In [ ]:
# Ex M05b.4: the values you found on paper
answer_M05b_4a = ...  # a) an integer
answer_M05b_4b = ...  # b) an integer
answer_M05b_4c = ...  # c) a number (2 decimals)
answer_M05b_4d = ...  # d) a number (4 decimals)
answer_M05b_4e = ...  # e) a number (1 decimal)
answer_M05b_4f = ...  # f) a number (3 decimals)
answer_M05b_4g = ...  # g) a number (2 decimals)

for letter, answer in zip("abcdefg", [answer_M05b_4a, answer_M05b_4b, answer_M05b_4c, answer_M05b_4d, answer_M05b_4e, answer_M05b_4f, answer_M05b_4g]):
    wbm.check(f"M05b.4{letter}", answer)

**Ex M05b.5 — Monter le plus vite : la direction et la valeur de la plus grande pente**

In [ ]:
# Ex M05b.5: the values you found on paper
answer_M05b_5a = ...  # a) a list of 2 integers
answer_M05b_5b = ...  # b) a number (3 decimals)
answer_M05b_5c = ...  # c) a number (2 decimals)
answer_M05b_5d = ...  # d) a number (3 decimals)
answer_M05b_5e = ...  # e) a number (2 decimals)
answer_M05b_5f = ...  # f) an irreducible fraction, as text

for letter, answer in zip("abcdef", [answer_M05b_5a, answer_M05b_5b, answer_M05b_5c, answer_M05b_5d, answer_M05b_5e, answer_M05b_5f]):
    wbm.check(f"M05b.5{letter}", answer)

**Ex M05b.6 — Le plan tangent et l'approximation d'une valeur**

In [ ]:
# Ex M05b.6: the values you found on paper
answer_M05b_6a = ...  # a) an integer
answer_M05b_6b = ...  # b) an integer
answer_M05b_6c = ...  # c) a number (2 decimals)
answer_M05b_6d = ...  # d) an integer
answer_M05b_6e = ...  # e) a number (3 decimals)
answer_M05b_6f = ...  # f) a number (4 decimals)
answer_M05b_6g = ...  # g) a number (4 decimals)

for letter, answer in zip("abcdefg", [answer_M05b_6a, answer_M05b_6b, answer_M05b_6c, answer_M05b_6d, answer_M05b_6e, answer_M05b_6f, answer_M05b_6g]):
    wbm.check(f"M05b.6{letter}", answer)

**Ex M05b.7 — Dérivées secondes croisées : le théorème de Schwarz vérifié**

In [ ]:
# Ex M05b.7: the values you found on paper
answer_M05b_7a = ...  # a) an expression in x and y, as text
answer_M05b_7b = ...  # b) an expression in x and y, as text
answer_M05b_7c = ...  # c) a number (4 decimals)
answer_M05b_7d = ...  # d) a number (4 decimals)
answer_M05b_7e = ...  # e) a number (4 decimals)
answer_M05b_7f = ...  # f) a number (3 decimals)
answer_M05b_7g = ...  # g) a number (3 decimals)

for letter, answer in zip("abcdefg", [answer_M05b_7a, answer_M05b_7b, answer_M05b_7c, answer_M05b_7d, answer_M05b_7e, answer_M05b_7f, answer_M05b_7g]):
    wbm.check(f"M05b.7{letter}", answer)

**Ex M05b.8 — Les points critiques et leur nature (test du second ordre)**

In [ ]:
# Ex M05b.8: the values you found on paper
answer_M05b_8a = ...  # a) an integer
answer_M05b_8b = ...  # b) a letter, as text
answer_M05b_8c = ...  # c) a letter, as text
answer_M05b_8d = ...  # d) an integer
answer_M05b_8e = ...  # e) an integer
answer_M05b_8f = ...  # f) an integer
answer_M05b_8g = ...  # g) an integer
answer_M05b_8h = ...  # h) a letter, as text

for letter, answer in zip("abcdefgh", [answer_M05b_8a, answer_M05b_8b, answer_M05b_8c, answer_M05b_8d, answer_M05b_8e, answer_M05b_8f, answer_M05b_8g, answer_M05b_8h]):
    wbm.check(f"M05b.8{letter}", answer)

**Ex M05b.9 — Pourquoi un petit pas contre le gradient fait baisser f : le calcul au premier ordre**

In [ ]:
# Ex M05b.9: the values you found on paper
answer_M05b_9a = ...  # a) a list of 2 integers
answer_M05b_9b = ...  # b) a list (2 decimals)
answer_M05b_9c = ...  # c) a number (3 decimals)
answer_M05b_9d = ...  # d) a number (2 decimals)
answer_M05b_9e = ...  # e) a number (3 decimals)
answer_M05b_9f = ...  # f) a number (3 decimals)
answer_M05b_9g = ...  # g) an irreducible fraction, as text

for letter, answer in zip("abcdefg", [answer_M05b_9a, answer_M05b_9b, answer_M05b_9c, answer_M05b_9d, answer_M05b_9e, answer_M05b_9f, answer_M05b_9g]):
    wbm.check(f"M05b.9{letter}", answer)

**Ex M05b.10 — argmin et argmax sur un segment**

In [ ]:
# Ex M05b.10: the values you found on paper
answer_M05b_10a = ...  # a) a number (1 decimal)
answer_M05b_10b = ...  # b) a list (1 decimal)
answer_M05b_10c = ...  # c) a number (1 decimal)
answer_M05b_10d = ...  # d) a list of 2 integers
answer_M05b_10e = ...  # e) an integer
answer_M05b_10f = ...  # f) a list of 2 integers
answer_M05b_10g = ...  # g) an integer
answer_M05b_10h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M05b_10a, answer_M05b_10b, answer_M05b_10c, answer_M05b_10d, answer_M05b_10e, answer_M05b_10f, answer_M05b_10g, answer_M05b_10h]):
    wbm.check(f"M05b.10{letter}", answer)

**Ex M05b.14 — La loss d'un neurone à deux poids sur deux exemples : son gradient**

In [ ]:
# Ex M05b.14: the values you found on paper
answer_M05b_14a = ...  # a) a number (3 decimals)
answer_M05b_14b = ...  # b) a number (1 decimal)
answer_M05b_14c = ...  # c) a number (1 decimal)
answer_M05b_14d = ...  # d) a list (2 decimals)
answer_M05b_14e = ...  # e) a number (4 decimals)
answer_M05b_14f = ...  # f) a list (4 decimals)

for letter, answer in zip("abcdef", [answer_M05b_14a, answer_M05b_14b, answer_M05b_14c, answer_M05b_14d, answer_M05b_14e, answer_M05b_14f]):
    wbm.check(f"M05b.14{letter}", answer)

**Ex M05b.16 — Lire une carte de lignes de niveau : estimer une dérivée partielle**

In [ ]:
# Ex M05b.16: the values you found on paper
answer_M05b_16a = ...  # a) an integer
answer_M05b_16b = ...  # b) an integer
answer_M05b_16c = ...  # c) an integer
answer_M05b_16d = ...  # d) an integer
answer_M05b_16e = ...  # e) an integer
answer_M05b_16f = ...  # f) an integer

for letter, answer in zip("abcdef", [answer_M05b_16a, answer_M05b_16b, answer_M05b_16c, answer_M05b_16d, answer_M05b_16e, answer_M05b_16f]):
    wbm.check(f"M05b.16{letter}", answer)

## Partie A · Ta librairie : dérivées partielles, dérivée directionnelle, plan tangent ; SymPy

Fiche §2 à §7. Les quatre fonctions de `mymath/multivar.py`, puis SymPy en plusieurs variables.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from wbm.nbtools import checked, drawn, run_pytest_group


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M05b.18 — Une dérivée partielle par différence centrée · `partial_derivative` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer une dérivée partielle par une différence centrée, en ne bougeant qu'une coordonnée d'une copie du point.  
**Prérequis :** fiche §2 ; M05a §5 · **mymath :** `multivar.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/multivar.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Une fonction de plusieurs variables prend **un seul** argument, un tableau NumPy `p` (`p[0]` est $x$, `p[1]` est $y$…) : par exemple `lambda p: p[0] ** 2 * p[1]`. NumPy suffit ; SymPy et `numpy.gradient` sont interdits : les tests comparent tes résultats aux dérivées exactes calculées par SymPy, et un test vérifie que tu ne te sers d'aucun des deux. Les nombres renvoyés sont des `float` de Python, les vecteurs des tableaux NumPy.

Écris `partial_derivative(f, point, i, h=1e-5)` (fiche §2) : $\frac{f(\mathbf{p} + h\,\mathbf{e}_i) - f(\mathbf{p} - h\,\mathbf{e}_i)}{2h}$, où $\mathbf{e}_i$ vaut 1 à la place $i$ et 0 ailleurs ; une `ValueError` si `i` n'est pas entre 0 et `len(point) - 1`, ou si $h \le 0$. Le point de l'appelant ne doit pas changer (relis le commentaire `TODO` de la fonction).

La vérification calcule :
- a) `partial_derivative(lambda p: p[0] ** 2 * np.exp(p[1]), (1.5, 0.0), 0)` (6 décimales) ;
- b) la même fonction, avec `i = 1` (6 décimales) ;
- c) `partial_derivative(lambda p: p[0] * p[1] * p[2], (2.0, -1.0, 4.0), 2)` (6 décimales) ;
- d) le nom de l'erreur levée par `partial_derivative(lambda p: p[0] + p[1] + p[2], (1.0, 2.0, 3.0), 3)` ;
- e) le nom de l'erreur levée par `partial_derivative(lambda p: p[0], (1.0, 2.0), 0, h=0)` ;
- f) `partial_derivative(lambda p: np.log(p[0] + p[1] ** 2), (1.0, 2.0), 1)` (4 décimales) ;
- g) le nom du type de `partial_derivative(lambda p: np.sin(p[0]), np.array([0.5]), 0)`, en texte.

Dans ta copie : calcule a), b), c) et f) à la main avec les règles de la fiche. Pourquoi les vérifications de `i` et de `h` doivent-elles passer **avant** tout calcul (regarde d) et e)) ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M05b")  # reload your saved file

In [ ]:
with checked("M05b.18a"):
    wbm.check("M05b.18a", mymath.multivar.partial_derivative(lambda p: p[0] ** 2 * np.exp(p[1]), (1.5, 0.0), 0),
              computed=True)
with checked("M05b.18b"):
    wbm.check("M05b.18b", mymath.multivar.partial_derivative(lambda p: p[0] ** 2 * np.exp(p[1]), (1.5, 0.0), 1),
              computed=True)
with checked("M05b.18c"):
    wbm.check("M05b.18c", mymath.multivar.partial_derivative(lambda p: p[0] * p[1] * p[2], (2.0, -1.0, 4.0), 2),
              computed=True)
with checked("M05b.18d"):
    wbm.check("M05b.18d", raised(mymath.multivar.partial_derivative, lambda p: p[0] + p[1] + p[2], (1.0, 2.0, 3.0), 3),
              computed=True)
with checked("M05b.18e"):
    wbm.check("M05b.18e", raised(mymath.multivar.partial_derivative, lambda p: p[0], (1.0, 2.0), 0, h=0),
              computed=True)
with checked("M05b.18f"):
    wbm.check("M05b.18f", mymath.multivar.partial_derivative(lambda p: np.log(p[0] + p[1] ** 2), (1.0, 2.0), 1),
              computed=True)
with checked("M05b.18g"):
    wbm.check("M05b.18g", type(mymath.multivar.partial_derivative(lambda p: np.sin(p[0]), np.array([0.5]), 0)).__name__,
              computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m05b_multivar.py", "test_partial_derivative_ or test_multivar_uses_")

### Ex M05b.19 — Dérivée directionnelle et direction de plus grande pente · `directional_derivative, steepest_ascent_direction` 🔨 ★★ ⏱️ 20 min
**Objectif :** programmer la dérivée directionnelle d'une fonction et la direction de plus grande montée, en normalisant les directions et en refusant les cas sans direction.  
**Prérequis :** fiche §3 et §4 ; Ex M05b.18 · **mymath :** `multivar.py` · **Parcours :** C, E

Écris (fiche §3 et §4) :
- `directional_derivative(f, point, direction, h=1e-5)` : la pente de `f` en `point` dans la direction du vecteur `direction`, **divisé d'abord par sa norme** ; une seule différence centrée le long du vecteur unitaire $\mathbf{u}$ suffit, $\frac{f(\mathbf{p} + h\mathbf{u}) - f(\mathbf{p} - h\mathbf{u})}{2h}$ (ou le produit scalaire du gradient avec $\mathbf{u}$) ; une `ValueError` si les longueurs diffèrent, si la direction est le vecteur nul, ou si $h \le 0$ ;
- `steepest_ascent_direction(f, point, h=1e-5)` : le vecteur unitaire $\frac{\nabla f(\mathbf{p})}{\|\nabla f(\mathbf{p})\|}$, en tableau NumPy, avec le gradient calculé par ta fonction `partial_derivative` (une coordonnée à la fois) ; une `ValueError` si la norme du gradient calculé est inférieure à $10^{-8}$ (un point critique n'a pas de direction de plus grande pente ; numériquement, le gradient y est minuscule, pas forcément exactement nul), ou si $h \le 0$.

La vérification calcule, pour $f(x, y) = x^2 y + 2y$ au point $(1, 3)$ :
- a) `directional_derivative(f, (1.0, 3.0), (1, 0))` (4 décimales) ;
- b) `directional_derivative(f, (1.0, 3.0), (2, 2))` (4 décimales) ;
- c) `directional_derivative(f, (1.0, 3.0), (-1, 2))` (4 décimales) ;
- d) `steepest_ascent_direction(f, (1.0, 3.0))` (une liste, 4 décimales) ;
- e) le nom de l'erreur levée par `directional_derivative(f, (1.0, 3.0), (0, 0))` ;

puis :
- f) le nom de l'erreur levée par `steepest_ascent_direction(lambda p: p[0] ** 2 + 2 * p[1] ** 2 + p[1] ** 3, (0.0, 0.0))` ;
- g) le nom du type de `steepest_ascent_direction(f, (1.0, 3.0))`, en texte ;
- h) la norme de `steepest_ascent_direction(lambda p: np.exp(p[0] - p[1]) + p[2] ** 2, (0.0, 1.0, 1.5))` (4 décimales).

Dans ta copie : calcule $\nabla f(1, 3)$ à la main, puis a), b) et c). Que représente la direction de c) sur la carte des lignes de niveau de $f$ ? Pourquoi f) n'a-t-il pas de réponse ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M05b")  # reload your saved file

In [ ]:
F_19 = lambda p: p[0] ** 2 * p[1] + 2 * p[1]                # noqa: E731
with checked("M05b.19a"):
    wbm.check("M05b.19a", mymath.multivar.directional_derivative(F_19, (1.0, 3.0), (1, 0)), computed=True)
with checked("M05b.19b"):
    wbm.check("M05b.19b", mymath.multivar.directional_derivative(F_19, (1.0, 3.0), (2, 2)), computed=True)
with checked("M05b.19c"):
    wbm.check("M05b.19c", mymath.multivar.directional_derivative(F_19, (1.0, 3.0), (-1, 2)), computed=True)
with checked("M05b.19d"):
    wbm.check("M05b.19d", np.asarray(mymath.multivar.steepest_ascent_direction(F_19, (1.0, 3.0))).tolist(),
              computed=True)
with checked("M05b.19e"):
    wbm.check("M05b.19e", raised(mymath.multivar.directional_derivative, F_19, (1.0, 3.0), (0, 0)), computed=True)
with checked("M05b.19f"):
    wbm.check("M05b.19f", raised(mymath.multivar.steepest_ascent_direction,
                                 lambda p: p[0] ** 2 + 2 * p[1] ** 2 + p[1] ** 3, (0.0, 0.0)), computed=True)
with checked("M05b.19g"):
    wbm.check("M05b.19g", type(mymath.multivar.steepest_ascent_direction(F_19, (1.0, 3.0))).__name__, computed=True)
with checked("M05b.19h"):
    wbm.check("M05b.19h", float(np.linalg.norm(mymath.multivar.steepest_ascent_direction(
        lambda p: np.exp(p[0] - p[1]) + p[2] ** 2, (0.0, 1.0, 1.5)))), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m05b_multivar.py", "test_directional_derivative_ or test_steepest_ascent_direction_ or test_multivar_uses_")

### Ex M05b.20 — Le plan tangent · `tangent_plane` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer le plan tangent à une surface en un point, et mesurer l'écart entre la surface et ce plan.  
**Prérequis :** fiche §5 ; Ex M05b.18 · **mymath :** `multivar.py` · **Parcours :** C

Écris `tangent_plane(f, point, h=1e-5)` (fiche §5) : les coefficients $(A, B, C)$ du plan tangent $z = Ax + By + C$ à la surface $z = f(x, y)$ au point $(x_0, y_0)$, avec $A$ et $B$ les deux dérivées partielles (ta fonction `partial_derivative`) et $C$ choisi pour que le plan passe par $(x_0, y_0, f(x_0, y_0))$ ; une `ValueError` si le point n'a pas exactement deux coordonnées, ou si $h \le 0$. Renvoie un tuple de trois `float` de Python.

La vérification calcule :
- a) `tangent_plane(lambda p: p[0] * np.exp(p[1]), (2.0, 0.0))` (une liste, 4 décimales) ;
- b) `tangent_plane(lambda p: np.log(p[0] ** 2 + p[1] ** 2), (1.0, 1.0))` (une liste, 4 décimales) ;
- c) le nom de l'erreur levée par `tangent_plane(lambda p: p[0] + p[1] + p[2], (1.0, 2.0, 3.0))` ;
- d) l'écart entre la surface de a) et son plan tangent au point $(2 ;\ 0{,}1)$, $f - (Ax + By + C)$ (5 décimales) ;
- e) les trois nombres renvoyés en a) sont-ils des `float` de Python ? (`True` ou `False`)

Dans ta copie : écris les deux plans à la main, sous la forme $z = f(x_0, y_0) + \ldots$, puis développés. L'écart de d) est-il positif ou négatif, et que dit son signe sur la surface près du point, dans la direction de $y$ ? Calcule $D = f_{xx}f_{yy} - f_{xy}^2$ en $(2, 0)$ : la surface reste-t-elle du même côté de son plan dans toutes les directions ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M05b")  # reload your saved file

In [ ]:
with checked("M05b.20a"):
    wbm.check("M05b.20a", list(mymath.multivar.tangent_plane(lambda p: p[0] * np.exp(p[1]), (2.0, 0.0))),
              computed=True)
with checked("M05b.20b"):
    wbm.check("M05b.20b", list(mymath.multivar.tangent_plane(lambda p: np.log(p[0] ** 2 + p[1] ** 2), (1.0, 1.0))),
              computed=True)
with checked("M05b.20c"):
    wbm.check("M05b.20c", raised(mymath.multivar.tangent_plane, lambda p: p[0] + p[1] + p[2], (1.0, 2.0, 3.0)),
              computed=True)
with checked("M05b.20d"):
    A_20, B_20, C_20 = mymath.multivar.tangent_plane(lambda p: p[0] * np.exp(p[1]), (2.0, 0.0))
    wbm.check("M05b.20d", 2 * math.exp(0.1) - (A_20 * 2 + B_20 * 0.1 + C_20), computed=True)
with checked("M05b.20e"):
    wbm.check("M05b.20e", all(type(v) is float for v in mymath.multivar.tangent_plane(
        lambda p: p[0] * np.exp(p[1]), (2.0, 0.0))), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m05b_multivar.py", "test_tangent_plane_ or test_multivar_uses_")

### Ex M05b.21 — SymPy : dérivées partielles, gradient, points critiques 📦 ★ ⏱️ 20 min
**Objectif :** calculer avec SymPy des dérivées partielles, un gradient, les points critiques et la matrice des dérivées secondes d'une fonction de deux variables, et les classer.  
**Prérequis :** fiche §2, §6 et §7 · **Parcours :** C, E

SymPy (M02a, M05a) dérive aussi en plusieurs variables :
- `x, y = sp.symbols("x y")`, puis `sp.diff(k, x)` donne $\frac{\partial k}{\partial x}$, et `sp.diff(k, x, y)` donne $\frac{\partial^2 k}{\partial y\,\partial x}$ ;
- `sp.solve([kx, ky], [x, y], dict=True)` résout le système $\nabla k = \mathbf{0}$ : une liste de dictionnaires `{x: ..., y: ...}` ;
- `sp.hessian(k, (x, y))` donne la matrice des dérivées secondes, `.det()` son déterminant $D$, et `.subs({x: 1, y: 2})` remplace les variables par des nombres.

Avec $k(x, y) = x^2 y + y^2 - y$, calcule :
- a) le nombre de points critiques de $k$ (un entier) ;
- b) la valeur de $k$ en son minimum local (une fraction irréductible, en texte) ;
- c) le déterminant $D$ de la matrice des dérivées secondes en $(1, 0)$ (un entier) ;
- d) $\frac{\partial^2 k}{\partial x\,\partial y}(3, 1)$ (un entier) ;
- e) $\nabla k(2, 1)$ (une liste de 2 entiers) ;
- f) $\frac{\partial k}{\partial y}$ (une expression en $x$ et $y$, en texte).

Dans ta copie : classe les points critiques par le test du second ordre (un tableau : point, $k_{xx}$, $k_{yy}$, $k_{xy}$, $D$, nature). Vérifie a) et c) à la main. Le minimum local est-il global (regarde $k(x, -1)$) ?

In [ ]:
answer_M05b_21a = ...  # a) an integer
answer_M05b_21b = ...  # b) an irreducible fraction, as text
answer_M05b_21c = ...  # c) an integer
answer_M05b_21d = ...  # d) an integer
answer_M05b_21e = ...  # e) a list of 2 integers
answer_M05b_21f = ...  # f) an expression in x and y, as text

In [ ]:
for letter, answer in zip("abcdef", [answer_M05b_21a, answer_M05b_21b, answer_M05b_21c, answer_M05b_21d,
                                     answer_M05b_21e, answer_M05b_21f]):
    wbm.check(f"M05b.21{letter}", answer)

## Partie B · Une prédiction, une expérience et une figure

Fiche §1, §4 et §5. La pente le long d'une ligne de niveau, l'erreur du plan tangent, la carte des lignes de niveau avec les flèches du gradient.

### Ex M05b.22 — Prédire : la dérivée directionnelle le long d'une ligne de niveau 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir la pente d'une fonction le long d'une ligne de niveau et le long du gradient, puis le vérifier.  
**Prérequis :** fiche §3 et §4 · **Parcours :** C, E

Soit $f(x, y) = x^2 + 3y^2$ et le point $P = (3, 1)$, où $f(P) = 12$ et $\nabla f(P) = (6, 6)$. On note $\mathbf{u} = \frac{(1, -1)}{\sqrt{2}}$, la direction de la ligne de niveau 12 en $P$. Choisis, pour chaque question :
- a) la dérivée directionnelle de $f$ en $P$ dans la direction $\mathbf{u}$ : (1) positive ; (2) nulle ; (3) négative ; (4) cela dépend du sens de $\mathbf{u}$ ;
- b) en avançant de 0,5 depuis $P$, en ligne droite dans la direction $\mathbf{u}$, $f$ : (1) reste exactement égale à 12 ; (2) augmente un peu ; (3) diminue un peu ; (4) ne peut pas se prévoir ;
- c) la dérivée directionnelle de $f$ en $P$ dans la direction du gradient : (1) 0 ; (2) $\|\nabla f(P)\| \approx 8{,}49$ ; (3) 1 ; (4) 6 ;
- d) combien de directions (vecteurs unitaires) donnent une dérivée directionnelle nulle en $P$ : (1) aucune ; (2) exactement une ; (3) exactement deux ; (4) une infinité.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience. Elle utilise la fonction `directional_derivative` de la référence (la tienne, M05b.19, fait la même chose).

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M05b_22a = ...  # a) 1, 2, 3 or 4, written BEFORE running the experiment below
prediction_M05b_22b = ...  # b) 1, 2, 3 or 4
prediction_M05b_22c = ...  # c) 1, 2, 3 or 4
prediction_M05b_22d = ...  # d) 1, 2, 3 or 4

In [ ]:
for letter, prediction in zip("abcd", [prediction_M05b_22a, prediction_M05b_22b, prediction_M05b_22c,
                                      prediction_M05b_22d]):
    wbm.check(f"M05b.22{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M05b_22a, prediction_M05b_22b, prediction_M05b_22c, prediction_M05b_22d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    ref_22 = wbm.load_mymath("ref", alias="mymath_ref")   # the reference library: the same results as yours
    f_22 = lambda p: p[0] ** 2 + 3 * p[1] ** 2          # noqa: E731
    P_22 = np.array([3.0, 1.0])
    grad_22 = np.array([2 * P_22[0], 6 * P_22[1]])            # (6, 6)
    tangent_22 = np.array([1.0, -1.0]) / math.sqrt(2)          # orthogonal to the gradient
    print("a) slope along the level line:", round(ref_22.multivar.directional_derivative(f_22, P_22, tangent_22), 6) + 0.0)
    print("b) f(P) =", f_22(P_22), "· f(P + 0.5 u) =", f_22(P_22 + 0.5 * tangent_22))
    print("c) slope along the gradient:", round(ref_22.multivar.directional_derivative(f_22, P_22, grad_22), 6),
          "· ||grad f|| =", round(float(np.linalg.norm(grad_22)), 6))
    angles_22 = np.linspace(0, 2 * np.pi, 3601)[:-1]
    slopes_22 = grad_22[0] * np.cos(angles_22) + grad_22[1] * np.sin(angles_22)
    print("d) angles (in degrees) where the slope is zero:",
          np.degrees(angles_22[np.isclose(slopes_22, 0, atol=1e-9)]).round(1))

**Après l'expérience**, dans ta copie : pourquoi la pente est-elle nulle dans la direction de la ligne de niveau, alors que $f$ augmente quand on avance en ligne droite (calcule $f(P + t\,\mathbf{u})$ en fonction de $t$) ? Dessine l'ellipse $x^2 + 3y^2 = 12$, sa tangente en $P$ et le gradient. Que deviennent ces réponses pour une fonction affine ?

### Ex M05b.23 — L'erreur du plan tangent quand on s'éloigne du point 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer comment l'écart entre une surface et son plan tangent grandit avec la distance au point, et selon la direction.  
**Prérequis :** fiche §5 ; Ex M05b.20 ; M05a §3 · **Parcours :** C

La fiche (§5) annonce que l'erreur de l'approximation par le plan tangent est d'ordre 2 : elle se comporte comme le carré de la distance au point. Vérifie-le.

Écris `plane_error(f, point, direction, r)`, qui renvoie l'écart, en valeur absolue, entre $f$ et son plan tangent en `point` (ta fonction `mymath.multivar.tangent_plane` : écris d'abord M05b.20), au point situé à la distance `r` de `point` dans la direction du vecteur `direction` (à normaliser), en `float`.

La vérification prend $f(x, y) = e^{x - y}$, dont le plan tangent en $(0, 0)$ est $z = 1 + x - y$, et calcule :
- a) l'erreur dans la direction $(1, 0)$, à la distance $r = 0{,}1$ (6 décimales) ;
- b) le rapport des erreurs pour $r = 0{,}2$ et $r = 0{,}1$, dans la direction $(1, 0)$ (2 décimales) ;
- c) la pente de la droite qui ajuste $\ln(\text{erreur})$ en fonction de $\ln r$, pour $r = 0{,}01$ ; 0,02 ; 0,04 ; 0,08 dans la direction $(1, 0)$, par `np.polyfit` (2 décimales) ;
- d) l'erreur dans la direction $(1, 1)$, à la distance $r = 0{,}5$ (6 décimales) ;
- e) l'erreur dans la direction $(1, -1)$, à la distance $r = 0{,}1$ (6 décimales).

Dans ta copie : que dit c) de l'ordre de l'erreur ? Que vaut d), et pourquoi, même loin du point (regarde $f$ le long de la droite $y = x$) ? Compare e) et a) avec le terme d'ordre 2 de Taylor, $\frac{s^2}{2}$, où $s = x - y$ au point d'arrivée.

In [ ]:
def plane_error(f, point, direction, r):
    """|f(q) - plane(q)| where q is at distance r from point in the direction of the vector direction, and plane is
    the tangent plane of f at point (mymath.multivar.tangent_plane)."""
    raise NotImplementedError("le vecteur unitaire, le point q, le plan tangent, puis l'écart en valeur absolue")

In [ ]:
F_23 = lambda p: math.exp(p[0] - p[1])                  # noqa: E731 - tangent plane at (0, 0): z = 1 + x − y
with checked("M05b.23a"):
    wbm.check("M05b.23a", plane_error(F_23, (0.0, 0.0), (1, 0), 0.1), computed=True)
with checked("M05b.23b"):
    wbm.check("M05b.23b", plane_error(F_23, (0.0, 0.0), (1, 0), 0.2) / plane_error(F_23, (0.0, 0.0), (1, 0), 0.1),
              computed=True)
with checked("M05b.23c"):
    rs_23 = np.array([0.01, 0.02, 0.04, 0.08])
    errors_23 = [plane_error(F_23, (0.0, 0.0), (1, 0), r) for r in rs_23]
    wbm.check("M05b.23c", np.polyfit(np.log(rs_23), np.log(errors_23), 1)[0], computed=True)
with checked("M05b.23d"):
    wbm.check("M05b.23d", plane_error(F_23, (0.0, 0.0), (1, 1), 0.5), computed=True)
with checked("M05b.23e"):
    wbm.check("M05b.23e", plane_error(F_23, (0.0, 0.0), (1, -1), 0.1), computed=True)

### Ex M05b.24 — Lignes de niveau et flèches du gradient sur la même figure 🎨 ★ ⏱️ 20 min
**Objectif :** tracer une carte de lignes de niveau et les flèches du gradient, et voir qu'elles coupent les lignes à angle droit et s'allongent là où les lignes se serrent.  
**Prérequis :** fiche §1 et §4 ; R0 (matplotlib) · **Parcours :** C, E

Écris `draw_gradient_map(ax, f, grad, xlim, ylim)`, qui dessine dans l'axe `ax`, sur le rectangle $[x_{\min}, x_{\max}] \times [y_{\min}, y_{\max}]$ donné par `xlim` et `ylim` :
- au moins 5 lignes de niveau de `f` (`ax.contour(X, Y, Z, levels=...)`, sur une grille fine faite par `np.meshgrid`), avec leurs valeurs (`ax.clabel`) ;
- les flèches du gradient sur une grille plus grossière, d'au moins 20 points répartis sur tout le rectangle (`ax.quiver(X, Y, U, V)`, où `U` et `V` sont les deux composantes du gradient) ;
- la même échelle sur les deux axes (`ax.set_aspect("equal")`) : sans elle, un cercle paraît ovale, et l'angle entre une flèche et une ligne de niveau est déformé.

Ici, `f(x, y)` et `grad(x, y)` prennent **deux** arguments, qui peuvent être des tableaux : `f` renvoie un tableau de valeurs, et `grad` un couple de tableaux `(U, V)`.

Puis trace la carte de $f(x, y) = \sin x + \cos y$, de gradient $(\cos x, -\sin y)$, sur $[-4, 4] \times [-4, 4]$.

Dans ta copie : repère sur ta figure les maxima, les minima et les points selles de $f$ (que font les flèches autour de chacun ?). Où les flèches sont-elles les plus longues, et que font les lignes de niveau à cet endroit ?

In [ ]:
def draw_gradient_map(ax, f, grad, xlim, ylim):
    """In ax, on the rectangle xlim × ylim: at least 5 level lines of f with their values, the arrows of grad on a
    coarser grid (at least 20 points), and the same scale on both axes."""
    raise NotImplementedError("np.meshgrid, ax.contour et ax.clabel, puis ax.quiver, et ax.set_aspect(\"equal\")")

In [ ]:
with checked("M05b.24 figure"):
    fig, ax = plt.subplots(figsize=(6.5, 6))
    try:
        created = drawn(draw_gradient_map, ax, lambda x, y: np.sin(x) + np.cos(y),
                        lambda x, y: (np.cos(x), -np.sin(y)), (-4, 4), (-4, 4))
    except BaseException:
        plt.close(fig)                                     # no empty figure while the function is not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex M05b.24 : draw_gradient_map ouvre sa propre figure (plt.figure ou plt.subplots) : dessine dans l'axe "
              "reçu.")
    else:
        plt.show()

**Vérifications automatiques** (sur un autre exemple ; la figure elle-même se juge à l'œil).

In [ ]:
with checked("M05b.24 vérifications"):
    import matplotlib.contour as mcontour
    import matplotlib.quiver as mquiver
    problems = []
    f_t = lambda x, y: np.exp(-(x ** 2 + 2 * y ** 2) / 4)                      # noqa: E731 - another example
    grad_t = lambda x, y: (-x / 2 * f_t(x, y), -y * f_t(x, y))                 # noqa: E731
    fig, ax = plt.subplots(figsize=(6, 5))
    try:
        created = drawn(draw_gradient_map, ax, f_t, grad_t, (-3, 3), (-2, 2))
    finally:
        plt.close(fig)                                                         # only the checks below are shown here
    if created:
        problems.append("draw_gradient_map ouvre sa propre figure : dessine dans l'axe reçu")
    else:
        contours = [c for c in ax.collections if isinstance(c, mcontour.ContourSet)]
        quivers = [c for c in ax.collections if isinstance(c, mquiver.Quiver)]
        if not contours:
            problems.append("il manque les lignes de niveau (ax.contour)")
        elif len(contours[0].levels) < 5:
            problems.append("trace au moins 5 lignes de niveau")
        elif not getattr(contours[0], "labelTexts", None):
            problems.append("écris la valeur des lignes de niveau (ax.clabel)")
        if not quivers:
            problems.append("il manque les flèches du gradient (ax.quiver)")
        else:
            q = quivers[0]
            xy = np.asarray(q.get_offsets(), dtype=float)
            u, v = np.ravel(q.U), np.ravel(q.V)
            gx, gy = grad_t(xy[:, 0], xy[:, 1])
            norms = np.hypot(gx, gy) * np.hypot(u, v)
            keep = norms > 1e-12
            cosines = (u * gx + v * gy)[keep] / norms[keep]
            if xy.shape[0] < 20:
                problems.append("dessine au moins 20 flèches, sur une grille")
            elif xy.shape[0] > 1000:
                problems.append("dessine les flèches sur une grille plus grossière que celle des lignes de niveau "
                                "(quelques centaines au plus) : sinon elles se recouvrent")
            elif cosines.size == 0 or np.min(cosines) < 0.999:
                problems.append("les flèches doivent avoir la direction et le sens du gradient en leur point de départ "
                                "(ax.quiver(X, Y, U, V), avec U et V les deux composantes du gradient)")
            if xy.size and (xy[:, 0].min() > -2.5 or xy[:, 0].max() < 2.5 or xy[:, 1].min() > -1.5
                            or xy[:, 1].max() < 1.5):
                problems.append("répartis les flèches sur tout le rectangle reçu")
        if ax.get_aspect() not in (1.0, "equal"):
            problems.append("prends la même échelle sur les deux axes (ax.set_aspect(\"equal\")) : sans elle, un "
                            "cercle paraît ovale et les angles sont déformés")
    if problems:
        print("❌ Ex M05b.24 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M05b.24 : lignes de niveau et flèches du gradient justes, à la même échelle.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §1 à §3 et §8. Trois erreurs à trouver, une grille de valeurs calculée d'un coup, puis une fonction retrouvée à partir de son gradient.

### Ex M05b.25 — Chasse à l'erreur : la mauvaise variable dérivée, une direction non unitaire 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans du code les erreurs classiques sur les dérivées partielles, les directions et le pas de descente, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §2, §3 et §8 · **Parcours :** C

Soit $f(x, y) = x^3 y - 2y^2$ et le point $P = (1, 3)$. Une collègue a écrit six lignes, chacune avec un commentaire qui dit ce qu'elle veut calculer ; chaque ligne utilise les variables des lignes précédentes telles qu'elles **devraient** être (on suppose les lignes précédentes justes). Trois lignes ne calculent pas ce que dit leur commentaire. **Sans exécuter de code**, trouve-les.

```python
x, y, h = 1.0, 3.0, 1e-5
dfdx = 3 * x ** 2 * y                                  # (A) ∂f/∂x at P
dfdy = (f(x + h, y) - f(x - h, y)) / (2 * h)           # (B) ∂f/∂y at P, by a centred difference
norm = math.sqrt(dfdx ** 2 + dfdy ** 2)                # (C) the norm of the gradient at P
slope = 3 * dfdx + 4 * dfdy                            # (D) the slope of f at P in the direction of (3, 4)
change = 0.01 * (dfdx + dfdy)                          # (E) about how much f changes from P to (1.01, 3.01)
x_new = x + 0.1 * dfdx                                 # (F) x after one step of gradient descent, η = 0.1
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) pour chacune des six lignes, de (A) à (F), la bonne valeur de ce que dit son commentaire (une liste de 6 nombres, 3 décimales).

Dans ta copie : pour chaque ligne fausse, quelle erreur a été faite, et comment la corriger ? Laquelle des lignes justes utilise l'approximation linéaire du §5 ?

In [ ]:
answer_M05b_25a = ...  # a) letters, as text
answer_M05b_25b = ...  # b) a list of 6 values (3 decimals)

In [ ]:
wbm.check("M05b.25a", answer_M05b_25a)
wbm.check("M05b.25b", answer_M05b_25b)

### Ex M05b.26 — Vectoriser : une fonction de deux variables sur une grille (`np.meshgrid`) 🛠️ ★ ⏱️ 15 min
**Objectif :** calculer les valeurs d'une fonction de deux variables sur toute une grille d'un coup, avec `np.meshgrid`, dans la bonne orientation.  
**Prérequis :** fiche §1 ; R0 (NumPy) · **Parcours :** C

Pour tracer une carte de lignes de niveau, il faut les valeurs de $f$ en chaque point d'une grille, souvent des centaines de milliers. Une double boucle en Python est lente ; NumPy calcule tout d'un coup, à condition de lui donner deux tableaux de même forme : `X, Y = np.meshgrid(xs, ys)` donne `X[j, i] = xs[i]` et `Y[j, i] = ys[j]`, de forme `(len(ys), len(xs))`. La ligne `j` correspond à `ys[j]` : c'est l'orientation qu'attendent `plt.contour` et `plt.imshow` (les $y$ en lignes, les $x$ en colonnes).

Écris `grid_values(f, xs, ys)`, qui renvoie le tableau `Z` de forme `(len(ys), len(xs))` tel que `Z[j, i] = f(xs[i], ys[j])`, pour une fonction `f(x, y)` écrite avec NumPy (qui accepte des tableaux). La vérification compare ta fonction à une double boucle, sur des grilles de tailles différentes en $x$ et en $y$ (une transposition s'y voit), puis une dernière cellule compare les temps de calcul.

Dans ta copie : pourquoi la vérification prend-elle des grilles de tailles différentes ? Que se passerait-il avec une fonction écrite avec `math.sin` au lieu de `np.sin` ? Combien de fois plus rapide est `grid_values` sur la grille de la dernière cellule ?

In [ ]:
def grid_values(f, xs, ys):
    """Z with shape (len(ys), len(xs)) and Z[j, i] = f(xs[i], ys[j]), for f(x, y) written with NumPy."""
    raise NotImplementedError("X, Y = np.meshgrid(xs, ys), puis f(X, Y)")

In [ ]:
with checked("M05b.26 vérifications"):
    problems = []
    xs_t, ys_t = np.linspace(-1, 2, 7), np.linspace(0, 1, 4)              # different lengths: a transposition shows
    for name, f_t in [("x + 10 y", lambda x, y: x + 10 * y), ("x² y − y", lambda x, y: x ** 2 * y - y),
                      ("cos(x y)", lambda x, y: np.cos(x * y))]:
        expected = np.array([[f_t(xv, yv) for xv in xs_t] for yv in ys_t])
        try:
            got = grid_values(f_t, xs_t, ys_t)
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            problems.append(f"grid_values pour {name} lève {type(exc).__name__} ({exc})")
            continue
        if got is None:
            problems.append("grid_values ne renvoie rien : il manque return")
            break
        got = np.asarray(got)
        if got.shape != expected.shape:
            problems.append(f"pour {name}, la forme attendue est (len(ys), len(xs)) = {expected.shape}, pas {got.shape}"
                            + (" : la ligne j correspond à ys[j] (np.meshgrid fait cela par défaut)"
                               if got.shape == expected.T.shape else ""))
            break
        elif not np.allclose(got, expected):
            problems.append(f"pour {name}, Z[j, i] doit valoir f(xs[i], ys[j])"
                            + (" : tu as échangé x et y" if np.allclose(got, np.array(
                                [[f_t(yv, xv) for xv in xs_t] for yv in ys_t]), equal_nan=True) else ""))
            break
    if problems:
        print("❌ Ex M05b.26 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M05b.26 : grid_values donne Z[j, i] = f(xs[i], ys[j]), avec la bonne forme, pour trois fonctions.")

**Le temps de calcul** (cette cellule utilise ta fonction).

In [ ]:
with checked("M05b.26 temps"):
    import time
    xs_26, ys_26 = np.linspace(-3, 3, 600), np.linspace(-2, 2, 400)
    f_26 = lambda x, y: np.sin(x) * np.exp(-y ** 2)          # noqa: E731
    start = time.perf_counter()
    Z_loop = np.array([[f_26(xv, yv) for xv in xs_26] for yv in ys_26])
    loop_seconds = time.perf_counter() - start
    start = time.perf_counter()
    Z_grid = np.asarray(grid_values(f_26, xs_26, ys_26))
    grid_seconds = time.perf_counter() - start
    same = Z_grid.shape == Z_loop.shape and np.allclose(Z_loop, Z_grid)
    print(f"double loop: {loop_seconds:.3f} s · grid_values: {grid_seconds:.4f} s · same values: {same}")
    if same and grid_seconds > loop_seconds / 5:
        print("💡 grid_values n'est guère plus rapide qu'une double boucle : calcule f sur toute la grille d'un coup.")

### Ex M05b.27 — Défi : retrouver une fonction du second degré à partir de son gradient 🏆 ★★★ ⏱️ 25 min
**Objectif :** retrouver les coefficients d'une fonction du second degré de deux variables à partir de son gradient, avec le moins d'évaluations possible, puis en tirer son point critique.  
**Prérequis :** fiche §2, §3 et §7 · **Parcours :** C

On ne connaît pas la fonction $f(x, y) = a x^2 + b\,xy + c y^2 + d x + e y + g$, mais on peut calculer son gradient en n'importe quel point, grâce à une fonction `grad(p)` qui renvoie le tableau $\nabla f(\mathbf{p})$ (et qui coûte cher : chaque appel compte).

Écris `fit_quadratic(grad)`, qui renvoie la liste `[a, b, c, d, e]` (la constante $g$ ne se voit pas dans le gradient). **L'objectif** : des coefficients justes à $10^{-9}$ près, avec **au plus 3 appels** à `grad`. Commence par écrire $\nabla f(x, y)$ en fonction des cinq coefficients, puis choisis des points où il se lit simplement.

La vérification essaie ta fonction sur six fonctions tirées au hasard (en comptant les appels), puis vérifie tes coefficients pour le gradient `GRAD_27` donné ci-dessous (une liste, 4 décimales).

Dans ta copie : pour la fonction de `GRAD_27`, écris $f$, trouve son point critique (un système de deux équations) et dis s'il s'agit d'un minimum, d'un maximum ou d'un point selle. Pourquoi deux appels ne peuvent-ils pas suffire en général ?

In [ ]:
def GRAD_27(p):
    """The gradient of an unknown function a x² + b xy + c y² + d x + e y + g."""
    x, y = float(p[0]), float(p[1])
    return np.array([3 * x - 2 * y + 0.5, -2 * x + 6 * y - 4])

In [ ]:
def fit_quadratic(grad):
    """The coefficients [a, b, c, d, e] of f = a x² + b xy + c y² + d x + e y + g, from its gradient grad(p), with at
    most 3 calls to grad."""
    raise NotImplementedError("écris ∇f en fonction de a, b, c, d, e, puis choisis trois points bien placés")

In [ ]:
with checked("M05b.27 défi"):
    rng_27 = np.random.default_rng(527)
    problems, worst_calls = [], 0
    for trial in range(6):
        coefs = rng_27.uniform(-3, 3, 5).round(2)

        def grad_t(p, coefs=coefs, calls=[]):
            calls.append(1)
            x, y = float(p[0]), float(p[1])
            a, b, c, d, e = coefs
            return np.array([2 * a * x + b * y + d, b * x + 2 * c * y + e])

        counter = grad_t.__defaults__[1]
        try:
            got = fit_quadratic(grad_t)
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            problems.append(f"fit_quadratic lève {type(exc).__name__} ({exc})")
            break
        worst_calls = max(worst_calls, len(counter))
        if np.shape(got) != (5,) or not np.allclose(got, coefs, atol=1e-9):
            problems.append(f"pour les coefficients {coefs.tolist()}, fit_quadratic renvoie {np.round(got, 6).tolist()}")
            break
    if problems:
        print("❌ Ex M05b.27 : " + problems[0] + ".")
    elif worst_calls > 3:
        print(f"❌ Ex M05b.27 : coefficients justes, mais {worst_calls} appels au gradient (objectif : 3 au plus) : "
              "choisis des points où le gradient se lit le plus simplement.")
    else:
        print(f"✅ Ex M05b.27 : défi réussi, les cinq coefficients de six fonctions retrouvés en {worst_calls} appels "
              "au plus.")
with checked("M05b.27"):
    wbm.check("M05b.27", list(np.asarray(fit_quadratic(GRAD_27), dtype=float)), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu lire une carte de lignes de niveau, et calculer des dérivées partielles et un gradient ?
2. Sais-tu calculer une dérivée directionnelle, et dire pourquoi le gradient donne la plus grande pente, à angle droit des lignes de niveau ?
3. Sais-tu écrire un plan tangent, et classer un point critique par le test du second ordre ?
4. Sais-tu expliquer pourquoi un petit pas contre le gradient fait baisser une fonction ?

**La suite** : le chapitre 5 de workbookIA (gradient numérique, lignes de niveau, descente de gradient), puis M06, la théorie de l'information.